# Laboration 2: Sökagenter

In this lab, the environment is fully observable. This means that the agent has complete knowledge of the world, i.e., it knows where all the food is located and where the walls are). This means that the agent can plan its path through the room before performing any actions. To do this, the agent needs keep track of how its actions will affect the world (i.e. maintain an internal state) and have some search strategy for exploring the search space to reach the desired goal state.

The lab is divided into two parts. In part 1, you will implement the `State` class used to represent and generate new states based on the actions performed by the agent. In part 2, you will implement a search strategy based on you implementation of `State`.

In preparation for this lab, especially part 2, we recommend reading Chapter 3 on search algorithms in *Artificial Intelligence: A Modern Approach*. A basic understanding of the principles underlying the classical search algorithms will be assumed throughout this lab.

As a reminder, the actions that can be performed by a Pacman agent are: 
- *GoForward*: Take one step forward
- *GoRight*: Turn 90 degrees right and take one step forward
- *GoLeft*: Turn 90 degrees left and take one step forward
- *GoBack*: Turn and 180 degrees and take one step forward
- *Stop*: Shut down the agent
- Any other command: No effect

*Note: If the the Pacman window freezes, you can still (usually) re-run the code. If this does not work, try `Kernel/Restart`, or restart the Jupyter Notebook* 

# Part 1: Implementing the State

Your task is to complete the class below to represent the state of the agent and the room. The state can be used to generate **new possible states** from the agent's current state. For example, assume that the agent is standing at a coordinate with no adjacent walls. From this state, the agent can move forward, backward, left, and right. Performing either of these actions should result in new states (not modifications of the current instance!).

The initial state contains the agent's starting position **(1,1)**, the agent's starting direction **(1,0)**, and a dictionary representing the map of the room. Directions are represented as tuples, indicating how the agents coordinates changes if the agent takes one step forward: (1,0) = east, (0,-1) = south, (-1,0) = west, (0,1) = north

In the map, coordinates (e.g. (1,1)) are used as keys and the values are either **w** (indicating a wall) or <b>*</b> (indicating food). Coordinates that are not represented in the map are assumed to be empty spaces.

For an action and some state, the corresponding **move** method returns a new instance of `State` that represents the state that would result if the action would be executed given the parent state (i.e., we are "thinking ahead"). It's important to consider, for instance, how the agent's position changes, if any food is eaten, and the actions that has led to the current state. If an action results in a collision with a wall, no new state should be generated (instead the method should return `None`).

This part can be a bit tricky, so it's a good idea to keep pen and paper handy.

## The Position API

We recommend that you use the Position API to help you manage position information. The `Position` class will help you find the coordinates that result from moving the agent in a given direction. It will also help you generate new directions when the agent turns. Have a look at the methods available in the API:

In [1]:
from positionUtil import Position
# the current position of the agent is x=2 and y=5
pos = (2,5)
# the agent is facing east, moving forward would add 1 to x and 0 to y
direction = (1,0)
# call the static helper method in the Position class
new_pos = Position.get_forward(pos, direction)
print(f"Moving east from {pos} results in {new_pos}\n")

help(Position)

Moving east from (2, 5) results in (3, 5)

Help on class Position in module positionUtil:

class Position(builtins.object)
 |  Directions are: (1,0) = east, (0,-1) = south, (-1,0) = west, (0,1) = north
 |  
 |  Methods defined here:
 |  
 |  get_back(position, direction)
 |      Get coordinate behind the current position relative to direction
 |  
 |  get_forward(position, direction)
 |      Get coordinate in front of the position relative to direction
 |  
 |  get_left(position, direction)
 |      Get coordinate to the left of the position relative to direction
 |  
 |  get_right(position, direction)
 |      Get coordinate to the right of the position relative to direction
 |  
 |  turn_left(direction)
 |      Returns the direction to the left
 |  
 |  turn_right(direction)
 |      Returns the direction to the right
 |  
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |  
 |  __dict__
 |      dictionary for instance variab

In [19]:
import copy
from positionUtil import *
from pacman import *
from agents import BaseAgent
from keyboardAgents import KeyboardAgent
from copy import deepcopy
import itertools
from collections import deque

class ModelBasedAgent(BaseAgent):
    
    class State(BaseAgent.State):
        def __init__(self, position, direction, room_map):
            
            self.position = position
            self.direction = direction
            self.room_map = room_map
            self.food_list = self.get_food()
            
            # print(f"position = {position}")
            # print(f"direction = {direction}")
            # print(f"room_map = {room_map}")

        def move_right(self):
            """Return the new state resulting from moving right"""
            print("-> GoRight")
            
            self.clockwise = True
            self.direction = self.get_new_direction()
            self.position = self.get_new_position()
            self.room_map[self.position] = "w"
            
            new_state = deepcopy(self)
            
            return new_state

        def move_left(self):
            """Return the new state resulting from moving left"""
            print("-> GoLeft")
            
            self.clockwise = False
            self.direction = self.get_new_direction()
            self.position = self.get_new_position()
            self.room_map[self.position] = "w"
            
            new_state = deepcopy(self)
            
            return new_state

        def move_forward(self):
            """Return the new state resulting from moving forward"""
            print("-> GoForward")
            
            self.direction = self.direction
            self.position = self.get_new_position()
            self.room_map[self.position] = "w"
            
            new_state = deepcopy(self)
            
            return new_state

        def move_back(self):
            """Return the new state resulting from moving backwards"""
            print("-> GoBack")
            
            self.direction = -self.direction[0], -self.direction[1]
            self.position = self.get_new_position()
            self.room_map[self.position] = "w"
            
            new_state = deepcopy(self)
            
            return new_state
        
        def get_new_position(self):
            """Get the position of the agent"""
            dir_x, dir_y = self.direction
            pos_x, pos_y = self.position
            return pos_x + dir_x, pos_y + dir_y
            
        
        def get_new_direction(self):
            """ Return the direction that the agent is facing"""
            
            if (self.direction == (0, 1) and self.clockwise) or (self.direction == (0, -1) and not self.clockwise):
                return (1, 0)
            if (self.direction == (0, -1) and self.clockwise) or (self.direction == (0, 1) and not self.clockwise):
                return (-1, 0)
            if (self.direction == (1, 0) and self.clockwise) or (self.direction == (-1, 0) and not self.clockwise):
                return (0, -1)
            if (self.direction == (-1, 0) and self.clockwise) or (self.direction == (1, 0) and not self.clockwise):
                return (0, 1)

        def get_direction(self):
            
            return self.direction
        
        def get_position(self):
            
            return self.position
            
        def get_food(self):
            """Get the position of all food as a list"""
            
            food_list = []
            
            for tile in self.room_map.keys():
                if self.room_map[tile] == "*":
                    food_list.append(tile)
            

            return food_list

        def get_actions(self):
            """Return all the actions neccessary to reach this state"""
            return "?"


    def search(self, start_state):
        """
        Search for a sequence of actions that will allow the pacman to
        devour all the food in the room
        """
        
        self.room_map = start_state.room_map
        
        shortest_path = self.breadth_first_nn(start_state.position, start_state.food_list)
        
        #shortest_path = self.a_star(start_state.position, start_state.food_list)

        print(shortest_path)


        directions_list = self.coords_to_directions(shortest_path)
        actions_list = self.directions_to_actions(directions_list)

        start_time = time.time()

        return actions_list



    def coords_to_directions(self, path):
        """Take a list of coordinates and return a list of actions required to reach them"""

        directions = []

        for i in range(len(path)-1):
            current_coord = path[i]
            next_coord = path[i+1]

            offset_x = current_coord[0] - next_coord[0]     # Calculate the amount of horizontal directions to be added by taking the difference between the x-coordinates
            offset_y = current_coord[1] - next_coord[1]     # Calculate the amount of vertical directions to be added by taking the difference between the y-coordinates
        
            if offset_x < 0:
                directions += -offset_x * [90]              # If the offset in the x-plane is negative, go east (90 degrees)
            else:
                directions += offset_x * [270]              # If the offset in the x-plane is positive, go west (270 degrees)
        
            if offset_y < 0:
                directions += -offset_y * [0]               # If the offset in the y-plane is negative, go north (0 degrees)
            else:
                directions += offset_y * [180]              # If the offset in the y-plane is positive, go south (180 degrees)
        
        return directions


    def directions_to_actions(self, directions):
        """Convert a list of directions to actions"""

        actions = []

        for i in range(len(directions)):
            
            direction = directions[i]
            
            if i == 0:                               # For the first iteration, we set last_direction to 90 because pacman is facing east
                last_direction = 90
            else:
                last_direction = directions[i-1]
                

            if direction == last_direction:                 # If the new direction is the same as the previous, go forward
                actions.append("GoForward")
            elif abs(direction - last_direction) == 180:    # If the new direction is opposite to the previous, go backward
                actions.append("GoBack")
            elif direction - last_direction == 90 or direction - last_direction == -270:    # If the new direction is clockwise 90 degrees to the previous turn right
                actions.append("GoRight")
            elif direction - last_direction == -90 or direction - last_direction == 270:    # If the new direction is counter-clockwise 90 degrees to the previous turn left
                actions.append("GoLeft")
        
        return actions


    def a_star(self, start_position, dots):

        open_list = deque()                          # Create the two empty lists of nodes
        closed_list = deque()
        
        first_dots = dots.copy()

        already_visited = []

        open_list.append((start_position, already_visited, dots))         # Node, Path to node

        while open_list:

            lowest_weight = 99999
            
            for i in range(len(open_list)):     # Iterate through the open list and select the node with the lowest weight
                

                potential_node = open_list[i][0]
                already_visited = open_list[i][1]
                weight = len(already_visited)

                print(open_list[i])
                print(weight)

                if weight < lowest_weight:
                    current_node = potential_node
                    current_path = already_visited
                    current_children = open_list[i][2]
                    lowest_weight = weight
            
            open_list.remove((current_node, current_path, current_children))
            closed_list.append((current_node, current_path, current_children))

            if not current_children:
                return_path = True
                for dot in first_dots:
                    if not dot in current_path:
                        return_path = False

                if return_path:
                    print(first_dots)
                    return current_path

            weighted_children = self.find_children_weights(current_node, current_children)
            
            for i in range(len(weighted_children)):

                child = weighted_children[i]

                child_node = child[0]
                child_path = child[1]
                child_children = child[2]
                child_cost = len(child[1])

                try:
                    if current_path[-1] == child_path[0]:
                        child_path.pop(0)
                except:
                    pass

                new_node = child_node, current_path + child_path, child_children

                if new_node in open_list:

                    existing_element_weight = self.check_open_list_weight(open_list, child_node)
                    print(existing_element_weight)
                    print(child_cost + lowest_weight)
                    
                    if existing_element_weight > child_cost + lowest_weight:
                        open_list.append(new_node)
                elif new_node in closed_list:
                    pass
                else:
                    open_list.append(new_node)

    def check_open_list_weight(self, open_list, node):
        
        for element in open_list:
            if open_list[0] == node:
                return len(open_list[1])
        
        return 99999
    
    def find_children_weights(self, node, children):
        
        try:
            children.remove(node)
        except ValueError:
            pass

        already_visited = set(node)                     # Save the nodes that have already been visited in a set (outside the searching tree)

        path_to_position = [node]                       # The path to the first position is a list with only the first position

        first_element = (node, path_to_position)        # The first element in the queue

        queue = deque()
        queue.append(first_element)                         # We have to fill the queue with something so the loop starts

        weighted_children = []

        while queue:

            position, path_to_position = queue.popleft()    # Assign the position and the path to it from the first element in the queue

            if not children:
                break

            for child in children:                                # If the queue finds a dot on the current position, return the position and the path to it
                if position == child:
                    
                    weighted_children.append((position, path_to_position, children))    

            neighbours = self.find_neighbours(position)     # Find the passable neighbours from the current position

            for neighbour in neighbours:                    # If the node has not already been visited, add the node to the queue
                if (not neighbour in path_to_position) and (not neighbour in already_visited):
                    next_element = (neighbour, path_to_position + [neighbour])
                    already_visited.add(neighbour)
                    queue.append(next_element)
        
        return weighted_children


# def heuristic(self, pos, next_pos):

    #     offset_x = pos[0] - next_pos[0]
    #     offset_y = pos[1] - next_pos[1]

    #     counter = 0
    #     for x in self.room_map.items():
    #         if x == "*":
    #             counter += 1

    #     cost = offset_x + offset_y + counter

    #     return cost


    def breadth_first_nn(self, position, dots, total_path=[]):
        """Nearest neighbour breadth first"""

        if not dots:                                        # Recursive stop condition, if there are no more dots, return the path to all of them
            return total_path + [position]
        
        else:

            next_position, path_to_position = self.find_shortest_path(position, dots)   # Find the shorterst path to the next dot

            dots.remove(next_position)

            total_path += path_to_position
            
            return self.breadth_first_nn(next_position, dots, total_path)               # Continue the recursion with the next dot as position


    def find_shortest_path(self, position, dots):
        """Find the closest dot from a position"""

        already_visited = set(position)                     # Save the nodes that have already been visited in a set (outside the searching tree)

        path_to_position = [position]                       # The path to the first position is a list with only the first position

        first_element = (position, path_to_position)        # The first element in the queue

        queue = deque()
        queue.append(first_element)                         # We have to fill the queue with something so the loop starts

        while queue:

            position, path_to_position = queue.popleft()    # Assign the position and the path to it from the first element in the queue

            for dot in dots:                                # If the queue finds a dot on the current position, return the position and the path to it
                if position == dot:
                    print(path_to_position)
                    return position, path_to_position

            neighbours = self.find_neighbours(position)     # Find the passable neighbours from the current position

            for neighbour in neighbours:                    # If the node has not already been visited, add the node to the queue
                if (not neighbour in path_to_position) and (not neighbour in already_visited):
                    next_element = (neighbour, path_to_position + [neighbour])
                    already_visited.add(neighbour)
                    queue.append(next_element)


    def find_neighbours(self, position):
        """Detect if neighbours are passable"""

        neighbours = []
        
        for direction in [(0, 1), (1, 0), (0, -1), (-1, 0)]:    # [north, east, south, west]
            
            dir_x = direction[0]                                # Get the offsets and position coordinates
            dir_y = direction[1]
            x = position[0]
            y = position[1]

            coord = (x+dir_x, y+dir_y)

            if coord in self.room_map.keys():                   # If the neighbour is a wall, don't add it, if it is something else, add it 
                if self.room_map[coord] != "w":
                    neighbours.append(coord)
            else:
                neighbours.append(coord)
        
        return neighbours


In [20]:
# Let the agent "search" for a solution
ModelBasedAgent.mode = "search"
# Enable/disable printing
ModelBasedAgent.printing = False

# Run ModelBasedAgent in the room layout "layouts/smallEmpty.lay"
args = readCommand(["--pacman", ModelBasedAgent,
                    "--layout", "mediumEmpty"])
runGames(**args)

hej
((1, 1), [], [(1, 2), (1, 7), (13, 7), (17, 1), (17, 6), (18, 4), (18, 7)])
0
hej
((1, 2), [(1, 1), (1, 2)], [(1, 2), (1, 7), (13, 7), (17, 1), (17, 6), (18, 4), (18, 7)])
2
((1, 7), [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7)], [(1, 2), (1, 7), (13, 7), (17, 1), (17, 6), (18, 4), (18, 7)])
7
((17, 1), [(1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1)], [(1, 2), (1, 7), (13, 7), (17, 1), (17, 6), (18, 4), (18, 7)])
17
((13, 7), [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (2, 7), (3, 7), (4, 7), (5, 7), (6, 7), (7, 7), (8, 7), (9, 7), (10, 7), (11, 7), (12, 7), (13, 7)], [(1, 2), (1, 7), (13, 7), (17, 1), (17, 6), (18, 4), (18, 7)])
19
((18, 4), [(1, 1), (1, 2), (1, 3), (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (7, 4), (8, 4), (9, 4), (10, 4), (11, 4), (12, 4), (13, 4), (14, 4), (15, 4), (16, 4), (17, 4), (18, 4)], [(1, 2), (1, 7), (13, 7), (17, 1), (17, 6), (

TypeError: object of type 'NoneType' has no len()

In [ ]:
# Control agent using keyboard arrows
ModelBasedAgent.mode = "keyboard"

# Print a representation of the current state when an action is actually executed
ModelBasedAgent.printing = True

# Run ModelBasedAgent in the room layout "layouts/custom.lay"
args = readCommand(["--pacman", ModelBasedAgent,
                    "--layout", "mediumEmpty"])
runGames(**args)

**Task 1**: Implement the methods in the `State` class above. Test your implementation by manually controlling the agent using the arrow keys on your keyboard. Verify that the reported position, direction, actions, and food locations match up with the printout of the agent's state.

**Task 2**: In your own words, describe the terms optimal and complete from an AI perspective. For each of the algorithms below, state whether they are optimal and/or complete. Assume that each algorithm is equipped with simple type of loop detection.

**Answer**: In regards to AI complete refers to a possible problem-solving strategy, which always finds a solution, given that there is one. Optimal refers to a complete solution, where all other possible solutions have been considered, that then is deemed the optimal one.

- breadth-first: Complete / Not optimal
- depth-first: Complete / Not optimal
- depth-limited: Complete, given that the goal is not below the limit / Not optimal
- iterative deepening: Complete / Optimal
- greedy: Complete / Not optimal
- A*: Complete / Optimal

**Task 3**: In your own words, describe what is meant by heuristics and what they are used for in AI. What is meant by an admissible heuristic?

**Answer**: Heurstics are time and cost effecient strategies to find solutions to a problem and are often used instead of less time and cost efficient calculations. Adopting a heuristic while solving a problem can be compared to using rules-of-thumb to get a good enough estimation or solution. Within AI heuristics are used in conjunction with other costly algorithms or problem-solving programs to estimate a good enough answer to a problem to save time and memory. 

An admissible heuristic refers to heuristics that are used in informed search algorithms that estimate the cost of reaching a goal-state, where the estimation is alwats lower than or equal to the actual cost of reaching the goal-state. Such heuristics needs to underestimate the cost to reach the goal node, otherwise an optimal solution cannot be found.

# Part 2

Run the agent in `search` mode. Try modifying the predefined list of actions returned by the `search` method.

**Task 3**: Draw a small example of the Pacman-world search tree down to depth 3 (starting from depth 0) with the same starting coordinate and direction as in the lab. Assume that all coordinates containing a 0 (e.g., (0,10) and (5,0)) are walls. Each node following the initial state should represent a new coordinate, and the edges should be labeled based on the action performed. Show only the "legal" actions and do not expand nodes that you consider to be duplicates of a previous state. Show this to your lab assistant to make sure that you have understood the principles of how a search tree is generated.

*Tip: To include a picture in your Jupyter Notebook, simply save your image to disk and reference it like this:* E.g.
![alt text](test.svg)

**Task 4**: Implement a breadth-first search algorithm in the `search` method above and return the list of actions required to eat all the food and then `Stop`. *Add comments to the code where appropriate*. Limit the number of states that can be explored to 10,000. A correct implementation should be able to solve all the easy and moderate layouts in the layout directory.

To successfully complete the task, you will need implement some type of loop control that allows you to ignore states that are equivalent to some previously explored state in the search. Without at least some basic loop control, even the easy problems will generate more than 10,000 states.

The predefined layouts have the following optimal solutions:
```
tinyEmpty       : optimal solution is 3 steps   (easy)
smallEmpty      : optimal solution is 8 steps   (easy)
smallLabyrinth  : optimal solution is 26 steps  (moderate)
mediumLabyrinth : optimal solution is 47 steps  (moderate)
mediumEmpty     : optimal solution is 32 steps  (moderate)
bigLabyrinth    : optimal solution is 134 steps (difficult)
bigEmpty        : optimal solution is 54 steps  (difficult)

Complexity of the rooms:
trivial         : can be solved without loop control
easy/moderate   : requires loop control
difficult       : requires heuristics to guide the search
```

**Task 6**: Briefly describe how your algorithm works. Imagine that you are explaining it to someone who doesn't have access to your code. Also describe how your loop detection works and how you decide if two states can be regarded equivalent.

**Answer**: The algorithm works by continuously expanding all nodes from the starting position, while expanding the nodes it checks whether they've previously been visited or if the node contains a wall. The expansion returns position of the food and a list of coordinates to follow to reach the food. When one iteration of the expansion is completed the node that is expanded changes to the returned node where food was found. This process continues until there is no food left on the map. Afterwards the path to follow is translated to directions for the agent to follow, which are then translated to corresponding actions.

**Task 5**: Is the new agent more intelligent than the agent that was implemented in the previous lab? Why or why not? Motivate your answer.

**Answer**: On a surface level the agent can be seen as more intelligent than the previously implemented one. By observation the agent is able to complete different mazes without any user input, which speaks for it's intelligence. However the reasoning, or whatever you call it, the agent uses to choose a path through the maze is not very complex. Because the pac-man world is extremely simplified, i.e the world is static and deterministic, the agent doesn't need to consider many factors while choosing a path through the maze it is faced with. 

Strictly comparing the current agent to the previous one, it's ways of choosing path to clear the map are definately more in line with what constitutes intelligence, i.e planning, translating plans to actions and considering obstructions (optimal way). 

**Task 6 (for VG):**: Adapt your solution to implement a heuristic A* search. The heuristic must be admissible. Compare the number of expanded nodes to your original breadth-first implementation; this should be a significant improvement. You should be able to solve at least one of the difficult layouts. Briefly describe your heuristic in your own words.

**Answer**: ...